# Bài 7.3 · Speech recognition: từ âm thanh ra chữ

**Câu hỏi của buổi này:** một giây âm thanh có 50 frame mà chỉ vài chữ, model biết chữ nào ở đâu bằng cách nào? Và con số WER đo được phụ thuộc những gì ngoài model?

Dữ liệu là **LibriSpeech**: sách nói tiếng Anh, phần `test-clean` (giọng dễ) và `test-other` (giọng khó hơn). Hai model cỡ gần bằng nhau: **wav2vec2-base-960h** (CTC, 95 triệu tham số) và **Whisper base.en** (encoder–decoder, 74 triệu tham số).

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập, nghe hai câu, waveform và log-mel |
| 2. CTC nhìn từng frame | xác suất từng ký tự theo thời gian, path có blank, gộp thành chữ |
| 3. CTC loss ba cách | liệt kê mọi path, quy hoạch động, `F.ctc_loss`: cùng một số |
| 4. Timestamp từ path | mỗi từ nằm ở đâu trên waveform |
| 5. WER có alignment | S/D/I của một câu sai thật |
| 6. Whisper và chuẩn hóa chữ | cùng một transcript, ba cách chấm, ba con số |
| 7. Gộp toàn tập | corpus WER và trung bình câu, loại lỗi, câu tệ nhất |
| 8. Thêm nhiễu | WER theo SNR, nghe thử |
| 9. Thời gian chạy | thời gian theo độ dài clip của hai kiểu model |
| 10. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 6 phút. Bài giảng: [ai-canuc80k · bài 7.3](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/7.3).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import io, re, time, itertools, urllib.request
from pathlib import Path
import soundfile as sf, librosa, torch, transformers
import torch.nn.functional as F
from IPython.display import Audio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, WhisperForConditionalGeneration, WhisperProcessor
transformers.logging.set_verbosity_error()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SR, N_PER, N_NOISE = 16000, 100, 40
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

def play(y, label):
    display(HTML(f'<div style="margin:6px 0 2px;font-weight:600">{label}</div>')); display(Audio(y, rate=SR))

## 1. Dữ liệu

Mỗi tập lấy ngẫu nhiên 100 câu dài không quá 20 giây. Reference của LibriSpeech viết hoa toàn bộ, không có dấu câu.

In [ ]:
root = Path('/tmp/librispeech'); root.mkdir(parents=True, exist_ok=True)
t = time.time()
data = {}
for split in ['test.clean', 'test.other']:
    f = root / f'{split}.parquet'
    if not f.exists():
        urllib.request.urlretrieve(f'https://huggingface.co/api/datasets/openslr/librispeech_asr/parquet/all/{split}/0.parquet', f)
    d = pd.read_parquet(f, columns=['audio', 'text', 'speaker_id', 'id'])
    meta = [sf.info(io.BytesIO(a['bytes'])) for a in d['audio']]
    d['giây'], d['sr'] = [m.duration for m in meta], [m.samplerate for m in meta]
    d = d[d['giây'] <= 20].sample(N_PER, random_state=0).reset_index(drop=True)
    d['wave'] = [sf.read(io.BytesIO(a['bytes']), dtype='float32')[0] for a in d['audio']]
    data[split] = d.drop(columns='audio')
t_load = time.time() - t
info = pd.DataFrame([(s, len(d), d['speaker_id'].nunique(), d['giây'].sum() / 60, d['text'].str.split().map(len).sum())
                     for s, d in data.items()], columns=['tập', 'số câu', 'số người nói', 'phút âm thanh', 'số từ reference'])
table(info, f'Hai tập LibriSpeech dùng trong notebook (tải và giải mã {t_load:.0f} giây)', fmt={'phút âm thanh': '{:.1f}'})
check('mọi clip là 16 kHz, một kênh', all((d['sr'] == SR).all() and all(w.ndim == 1 for w in d['wave']) for d in data.values()))

clean, other = data['test.clean'], data['test.other']
ex = clean.loc[clean['giây'].between(4, 7)].index[0]
play(clean.at[ex, 'wave'], f'test-clean: "{clean.at[ex, "text"]}"')
play(other.at[0, 'wave'], f'test-other: "{other.at[0, "text"]}"')

w = clean.at[ex, 'wave']
fig, axes = plt.subplots(2, 1, figsize=(13, 5.6), sharex=True, gridspec_kw={'height_ratios': [1, 1.6]})
axes[0].plot(np.arange(len(w)) / SR, w, color=C['blue'], lw=.5); axes[0].set_ylabel('biên độ')
mel = librosa.power_to_db(librosa.feature.melspectrogram(y=w, sr=SR, n_fft=400, hop_length=160, n_mels=80), ref=np.max)
axes[1].imshow(mel, origin='lower', aspect='auto', cmap='magma', extent=(0, len(w) / SR, 0, 80))
axes[1].set_ylabel('dải mel'); axes[1].set_xlabel('giây')
fig.suptitle(f'Câu test-clean: {len(w)} số ở 16 kHz, {len(clean.at[ex, "text"])} ký tự transcript', y=0.97)
plt.tight_layout(); plt.show()

## 2. CTC nhìn từng frame

wav2vec2 cắt âm thanh thành frame 20 ms (bước 320 mẫu) và cho mỗi frame một phân phối trên 32 nhãn: 26 chữ cái, dấu nháy, dấu cách từ `|`, và **blank** (ở đây là `<pad>`, viết `_`). Đọc chữ bằng quy tắc của bài học: lấy nhãn có xác suất cao nhất mỗi frame, **gộp nhãn trùng liền kề, rồi bỏ blank**.

In [ ]:
with quiet():
    w2v_proc = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
    w2v = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h').to(device).eval()
vocab = {i: s for s, i in w2v_proc.tokenizer.get_vocab().items()}
BLANK = w2v_proc.tokenizer.pad_token_id

def w2v_logprobs(wave):
    x = w2v_proc(wave, sampling_rate=SR, return_tensors='pt').input_values.to(device)
    with torch.no_grad(): return w2v(x).logits[0].float().log_softmax(-1).cpu()

def collapse(path):
    out = [k for j, k in enumerate(path) if k != BLANK and (j == 0 or k != path[j - 1])]
    return ''.join(vocab[k] for k in out).replace('|', ' ').strip()

lp = w2v_logprobs(w)
path = lp.argmax(-1).tolist()
T, U = lp.shape[0], len(clean.at[ex, 'text'])
table(pd.DataFrame([('frame T', T), ('ký tự transcript U', U), ('frame là blank', f'{np.mean(np.array(path) == BLANK):.0%}')],
                   columns=['đại lượng', 'giá trị']), f'Một câu {len(w) / SR:.1f} giây')

n = 60
fig, ax = plt.subplots(figsize=(14, 4.2))
show_ids = sorted(set(path[:n]))
ax.imshow(lp[:n, show_ids].exp().T.numpy(), aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_yticks(range(len(show_ids)), ['_' if k == BLANK else vocab[k] for k in show_ids])
ax.set_xticks(range(n), ['_' if k == BLANK else vocab[k] for k in path[:n]], fontsize=8, family='monospace')
ax.set_xlabel('nhãn được chọn ở từng frame (20 ms mỗi frame)'); ax.set_ylabel('nhãn')
ax.set_title(f'Xác suất từng nhãn trong {n} frame đầu ({n * 0.02:.1f} giây): đậm là gần 1')
plt.tight_layout(); plt.show()

shown = ''.join('_' if k == BLANK else vocab[k] for k in path[:n])
table(pd.DataFrame([('path (60 frame đầu)', shown), ('gộp trùng, bỏ blank', collapse(path[:n])),
                    ('cả câu sau khi gộp', collapse(path)), ('reference', clean.at[ex, 'text'])],
                   columns=['bước', 'chuỗi']), 'Từ path tới chữ')
check('số frame lớn hơn hẳn số ký tự', T > 2 * U)
note(f'{np.mean(np.array(path) == BLANK):.0%} số frame là blank: model chỉ "bật" chữ ở vài frame rồi im. Không ai dạy nó chữ nào nằm ở frame nào; CTC chỉ cho transcript cả câu.')

## 3. CTC loss ba cách

$P(y\mid X)$ là tổng xác suất của **mọi** path gộp về $y$. Với 5 frame và 4 nhãn chỉ có $4^5 = 1024$ path, liệt kê được hết. Với câu thật thì số path khổng lồ, phải dùng quy hoạch động trên chuỗi mở rộng `_ c _ a _ t _` như bài học. Ba cách phải ra cùng một số.

In [ ]:
def ctc_dp(logp, target, blank):
    ext = [blank] + sum([[c, blank] for c in target], [])
    S = len(ext)
    a = np.full(S, -np.inf); a[0] = logp[0, ext[0]]; a[1] = logp[0, ext[1]]
    skip = np.array([s >= 2 and ext[s] != blank and ext[s] != ext[s - 2] for s in range(S)])
    for t in range(1, len(logp)):
        b = a.copy()
        b[1:] = np.logaddexp(b[1:], a[:-1])
        b[2:] = np.where(skip[2:], np.logaddexp(b[2:], a[:-2]), b[2:])
        a = b + logp[t, ext]
    return -np.logaddexp(a[-1], a[-2])

# đồ chơi: 5 frame, nhãn {_, c, a, t} = {0, 1, 2, 3}, transcript "cat" = [1, 2, 3]
torch.manual_seed(0)
toy = torch.randn(5, 4).log_softmax(-1).double()
total, n_ok = 0.0, 0
for p in itertools.product(range(4), repeat=5):
    if [k for j, k in enumerate(p) if k != 0 and (j == 0 or k != p[j - 1])] == [1, 2, 3]:
        total += float(sum(toy[t, k] for t, k in enumerate(p)).exp()); n_ok += 1
ref_toy = F.ctc_loss(toy[:, None], torch.tensor([[1, 2, 3]]), [5], [3], blank=0, reduction='sum').item()
toy_rows = [('liệt kê mọi path', -np.log(total)), ('quy hoạch động', ctc_dp(toy.numpy(), [1, 2, 3], 0)), ('F.ctc_loss', ref_toy)]

# câu thật: log-prob của wav2vec2, transcript reference
tok = lambda s: [w2v_proc.tokenizer.get_vocab()[ch] for ch in s.replace(' ', '|')]
y_true = tok(clean.at[ex, 'text'])
real_dp = ctc_dp(lp.double().numpy(), y_true, BLANK)
real_ref = F.ctc_loss(lp[:, None], torch.tensor([y_true]), [T], [len(y_true)], blank=BLANK, reduction='sum').item()
words = clean.at[ex, 'text'].split(); wrong = ' '.join(words[:-1] + ['APPLE'])
real_wrong = ctc_dp(lp.double().numpy(), tok(wrong), BLANK)
table(pd.DataFrame(toy_rows, columns=['cách tính', '−log P("cat")']), f'Đồ chơi: {n_ok} trong 1024 path gộp về "cat"', fmt={'−log P("cat")': '{:.6f}'})
table(pd.DataFrame([('quy hoạch động (numpy)', 'reference', real_dp), ('F.ctc_loss', 'reference', real_ref),
                    ('quy hoạch động (numpy)', f'đổi từ cuối thành APPLE', real_wrong)],
                   columns=['cách tính', 'transcript', 'loss']), f'Câu thật: T = {T} frame, {len(y_true)} nhãn', fmt={'loss': '{:.3f}'})
check('ba cách cho cùng loss trên bài đồ chơi', abs(toy_rows[0][1] - toy_rows[1][1]) < 1e-5 and abs(toy_rows[1][1] - ref_toy) < 1e-5)
check('quy hoạch động khớp F.ctc_loss trên câu thật', abs(real_dp - real_ref) < 1e-2)
check('transcript sai có loss lớn hơn', real_wrong > real_dp + 5)
note(f'Trên câu thật, đổi một từ cuối làm loss tăng từ {real_dp:.2f} lên {real_wrong:.1f}: model gần như chắc chắn transcript nào đúng. '
     f'Quy hoạch động chỉ tốn T × (2U + 1) = {T} × {2 * len(y_true) + 1} ô, thay cho việc liệt kê số path khổng lồ.')

## 4. Timestamp từ path

Path greedy cho biết frame nào "bật" chữ nào, nên mỗi từ có thời điểm bắt đầu và kết thúc (mỗi frame 20 ms). Không ai đánh dấu ranh giới từ cho model; vị trí tự lộ ra từ CTC.

In [ ]:
def word_spans(path):
    spans, cur, start, last = [], '', None, None
    for t, k in enumerate(path):
        if k == BLANK: continue
        ch = vocab[k]
        if ch == '|':
            if cur: spans.append((cur, start, last)); cur = ''
            continue
        if not cur: start = t
        if not (t > 0 and path[t - 1] == k): cur += ch
        last = t
    if cur: spans.append((cur, start, last))
    return [(s, a * 0.02, (b + 1) * 0.02) for s, a, b in spans]

spans = word_spans(path)
fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(np.arange(len(w)) / SR, w, color=C['gray'], lw=.4)
top = np.abs(w).max()
for j, (s, a, b) in enumerate(spans):
    ax.axvspan(a, b, color=C['clay'] if j % 2 else C['blue'], alpha=.18)
    ax.text((a + b) / 2, top * (1.05 if j % 2 else 1.25), s.lower(), ha='center', fontsize=9)
ax.set_ylim(-top * 1.1, top * 1.45); ax.set_xlabel('giây'); ax.set_yticks([])
ax.set_title('Mỗi từ ứng với một đoạn waveform, lấy từ frame bật chữ của CTC')
plt.tight_layout(); plt.show()
check('số từ trong path bằng số từ của transcript greedy', len(spans) == len(collapse(path).split()))
note(f'{len(spans)} từ nằm trong {spans[-1][2] - spans[0][1]:.1f} giây; khoảng trống giữa các từ là các frame blank. Đây là cách cắt clip dài, gắn phụ đề theo từ mà không cần nhãn thời gian.')

## 5. WER có alignment

WER $= (S + D + I) / N_{ref}$ trên một alignment tối ưu, tìm bằng quy hoạch động edit distance. Hàm dưới đây lưu cả đường đi để in từng thao tác. Lấy câu test-other mà wav2vec2 chép sai nhiều nhất để xem.

In [ ]:
def align(ref, hyp):
    r, h = ref.split(), hyp.split()
    D = np.zeros((len(r) + 1, len(h) + 1), int); D[:, 0] = range(len(r) + 1); D[0, :] = range(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    ops, i, j = [], len(r), len(h)
    while i or j:
        if i and j and D[i, j] == D[i - 1, j - 1] + (r[i - 1] != h[j - 1]):
            ops.append(('=' if r[i - 1] == h[j - 1] else 'S', r[i - 1], h[j - 1])); i, j = i - 1, j - 1
        elif i and D[i, j] == D[i - 1, j] + 1: ops.append(('D', r[i - 1], '')); i -= 1
        else: ops.append(('I', '', h[j - 1])); j -= 1
    return ops[::-1]

def counts(ref, hyp):
    ops = align(ref, hyp)
    return {k: sum(o[0] == k for o in ops) for k in 'SDI'} | {'N': len(ref.split())}

def show_align(ops):
    col = {'=': 'inherit', 'S': C['clay'], 'D': C['blue'], 'I': C['sand']}
    cells = ''.join(f'<td style="padding:2px 6px;text-align:center;color:{col[o]};border-bottom:2px solid {col[o] if o != "=" else "transparent"}">'
                    f'{a or "·"}<br><span style="opacity:.75">{b or "·"}</span><br><b>{o if o != "=" else ""}</b></td>' for o, a, b in ops)
    display(HTML(f'<div style="overflow-x:auto"><table style="border-collapse:collapse;font-family:monospace;font-size:12px"><tr>{cells}</tr></table></div>'))

t = time.time()
for d in data.values():
    d['w2v'] = [collapse(w2v_logprobs(x).argmax(-1).tolist()) for x in d['wave']]
t_w2v = time.time() - t
c = [counts(r, h) for r, h in zip(other['text'], other['w2v'])]
worst = int(np.argmax([(k['S'] + k['D'] + k['I']) / k['N'] for k in c]))
ops = align(other.at[worst, 'text'], other.at[worst, 'w2v'])
show_align(ops)
k = c[worst]
table(pd.DataFrame([(k['S'], k['D'], k['I'], k['N'], (k['S'] + k['D'] + k['I']) / k['N'])], columns=['S thay', 'D thiếu', 'I thừa', 'N từ reference', 'WER']),
      'Câu test-other wav2vec2 chép tệ nhất (hàng trên: reference, hàng dưới: model)', fmt={'WER': '{:.3f}'})
play(other.at[worst, 'wave'], 'nghe câu này')
toy_ops = align('A B C D E F G H I J', 'A X C D F G H I J K')
check('ví dụ của bài học: thay 1, thiếu 1, thừa 1 trên 10 từ cho WER 0.3', sum(o[0] != '=' for o in toy_ops) / 10 == 0.3)

## 6. Whisper và chuẩn hóa chữ

Whisper là encoder–decoder: sinh từng token, có chữ hoa, dấu câu, chữ số. Reference của LibriSpeech thì viết hoa, không dấu câu, số viết bằng chữ. Nếu so thẳng, mọi dấu phẩy và chữ hoa đều thành lỗi. Ba cách chấm:

- **thô**: so nguyên chuỗi, chỉ tách khoảng trắng;
- **viết thường, bỏ dấu câu**;
- **bộ chuẩn hóa tiếng Anh của Whisper**: còn đổi số, cách viết Anh–Mỹ, từ viết tắt (dấu câu nó để sót, như `...`, được bỏ nốt).

Phải áp dụng **cùng một** cách cho cả reference lẫn transcript, và cho cả hai model.

In [ ]:
with quiet():
    wh_proc = WhisperProcessor.from_pretrained('openai/whisper-base.en')
    wh = WhisperForConditionalGeneration.from_pretrained('openai/whisper-base.en').to(device).eval()

def whisper(wave):
    feats = wh_proc(wave, sampling_rate=SR, return_tensors='pt').input_features.to(device)
    with torch.no_grad(): ids = wh.generate(feats, max_new_tokens=200)
    return wh_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()

times = []
for split, d in data.items():
    out = []
    for x in d['wave']:
        if device == 'cuda': torch.cuda.synchronize()
        t = time.time(); out.append(whisper(x))
        if device == 'cuda': torch.cuda.synchronize()
        times.append((split, len(x) / SR, time.time() - t))
    d['whisper'] = out

def basic(s): return re.sub(r"[^a-z' ]+", ' ', s.lower()).split()
norm_ways = {'thô': lambda s: s.split(), 'viết thường, bỏ dấu câu': basic, 'bộ chuẩn hóa của Whisper': lambda s: basic(wh_proc.tokenizer.normalize(s))}

def corpus_wer(refs, hyps, f):
    c = [counts(' '.join(f(r)), ' '.join(f(h))) for r, h in zip(refs, hyps)]
    return sum(k['S'] + k['D'] + k['I'] for k in c) / sum(k['N'] for k in c)

rows = []
for nm, f in norm_ways.items():
    for model in ['w2v', 'whisper']:
        rows.append((nm, {'w2v': 'wav2vec2 (CTC)', 'whisper': 'Whisper base.en'}[model],
                     corpus_wer(clean['text'], clean[model], f), corpus_wer(other['text'], other[model], f)))
wer = pd.DataFrame(rows, columns=['cách chấm', 'model', 'WER test-clean', 'WER test-other'])
i = int(np.flatnonzero(clean['whisper'].str.contains(r'[,.]'))[0])
table(pd.DataFrame([('reference', clean.at[i, 'text']), ('Whisper', clean.at[i, 'whisper']), ('wav2vec2', clean.at[i, 'w2v'])],
                   columns=['', 'transcript']), 'Cùng một câu, hai model viết theo hai kiểu')
table(wer, 'Corpus WER: cùng transcript, ba cách chấm', fmt={c: '{:.1%}' for c in wer.columns[2:]})
wv = wer.set_index(['cách chấm', 'model'])
w_raw, w_norm = wv.loc[('thô', 'Whisper base.en'), 'WER test-clean'], wv.loc[('bộ chuẩn hóa của Whisper', 'Whisper base.en'), 'WER test-clean']
check('chuẩn hóa làm WER của Whisper giảm mạnh', w_norm < w_raw / 3)
nz = wv.xs('bộ chuẩn hóa của Whisper', level=0)
note(f'Chấm thô, Whisper có WER {w_raw:.0%} trên test-clean, vì dấu câu và chữ hoa đều tính là sai; chuẩn hóa xong chỉ còn <b>{w_norm:.1%}</b>. '
     f'Chấm cùng một cách thì hai model mới so được: test-clean {nz.at["wav2vec2 (CTC)", "WER test-clean"]:.1%} và {nz.at["Whisper base.en", "WER test-clean"]:.1%}, '
     f'test-other {nz.at["wav2vec2 (CTC)", "WER test-other"]:.1%} và {nz.at["Whisper base.en", "WER test-other"]:.1%} (wav2vec2 trước, Whisper sau). '
     'Đề thi ghi scorer nào thì chuẩn hóa đúng như scorer đó.')

## 7. Gộp toàn tập, rồi đọc loại lỗi

Từ đây chấm bằng bộ chuẩn hóa của Whisper. **Corpus WER** cộng lỗi mọi câu rồi chia tổng số từ reference; **trung bình câu** lấy WER từng câu rồi tính trung bình, nên câu ngắn nặng ký hơn.

In [ ]:
NORM = norm_ways['bộ chuẩn hóa của Whisper']
rows, worst_rows = [], []
for split, d in data.items():
    for model, name in [('w2v', 'wav2vec2'), ('whisper', 'Whisper')]:
        c = pd.DataFrame([counts(' '.join(NORM(r)), ' '.join(NORM(h))) for r, h in zip(d['text'], d[model])])
        err = c[['S', 'D', 'I']].sum(1)
        rows.append((split, name, err.sum() / c['N'].sum(), (err / c['N']).mean(), c['S'].sum(), c['D'].sum(), c['I'].sum(), (err == 0).mean()))
        if split == 'test.other':
            j = int((err / c['N']).idxmax()); worst_rows.append((name, d.at[j, 'text'].lower(), d.at[j, model], err[j] / c['N'][j]))
agg = pd.DataFrame(rows, columns=['tập', 'model', 'corpus WER', 'trung bình câu', 'S', 'D', 'I', 'câu đúng hết'])
table(agg, 'Hai cách gộp và ba loại lỗi', fmt={'corpus WER': '{:.1%}', 'trung bình câu': '{:.1%}', 'câu đúng hết': '{:.0%}'})
table(pd.DataFrame(worst_rows, columns=['model', 'reference', 'transcript', 'WER câu']), 'Câu test-other bị chép tệ nhất của mỗi model', fmt={'WER câu': '{:.2f}'})
a = agg.set_index(['tập', 'model'])
check('test-other khó hơn test-clean với cả hai model', all(a.loc[('test.other', m), 'corpus WER'] > a.loc[('test.clean', m), 'corpus WER'] for m in ['wav2vec2', 'Whisper']))
gap = (a['trung bình câu'] - a['corpus WER']).abs().max()
note(('Lỗi thay từ (S) nhiều hơn thiếu và thừa cộng lại ở mọi dòng. ' if (agg['S'] > agg['D'] + agg['I']).all() else '') + f'test-other có WER cao hơn test-clean khoảng {a.xs("test.other")["corpus WER"].mean() / a.xs("test.clean")["corpus WER"].mean():.1f} lần với cả hai model. '
     f'Corpus WER và trung bình câu lệch nhau tới {gap * 100:.1f} điểm phần trăm: nói "WER" mà không nói cách gộp thì chưa đủ.')

## 8. Thêm nhiễu

Cộng nhiễu trắng vào 40 câu test-clean ở các mức SNR (tỉ số tín hiệu trên nhiễu, tính bằng dB): 20 dB là nhiễu nhẹ, 0 dB là nhiễu to bằng tiếng nói.

In [ ]:
def add_noise(x, snr_db, rng):
    noise = rng.standard_normal(len(x)).astype(np.float32)
    return x + noise * np.sqrt((x ** 2).mean() / (10 ** (snr_db / 10)) / (noise ** 2).mean())

SNRS = [None, 20, 10, 5, 0]
sub = clean.iloc[:N_NOISE]
rows = []
for snr in SNRS:
    rng = np.random.default_rng(0)
    waves = list(sub['wave']) if snr is None else [add_noise(x, snr, rng) for x in sub['wave']]
    hw = [collapse(w2v_logprobs(x).argmax(-1).tolist()) for x in waves]
    hh = [whisper(x) for x in waves]
    rows.append(('sạch' if snr is None else f'{snr} dB', corpus_wer(sub['text'], hw, NORM), corpus_wer(sub['text'], hh, NORM)))
    if snr == 5: noisy_example, noisy_pair = waves[0], (hw[0], hh[0])
noise = pd.DataFrame(rows, columns=['SNR', 'wav2vec2', 'Whisper'])
fig, ax = plt.subplots(figsize=(9, 4.4))
xs = np.arange(len(noise))
for col, colr, dy in [('wav2vec2', C['blue'], 8), ('Whisper', C['clay'], -15)]:
    ax.plot(xs, noise[col] * 100, 'o-', color=colr, lw=2, label=col)
    for x_, v in zip(xs, noise[col]): ax.annotate(f'{v:.0%}', (x_, v * 100), textcoords='offset points', xytext=(0, dy), ha='center', fontsize=9, color=colr)
ax.set_xticks(xs, noise['SNR']); ax.set_xlabel('mức nhiễu'); ax.set_ylabel('corpus WER (%)'); ax.legend(frameon=False)
ax.set_title(f'{N_NOISE} câu test-clean, nhiễu trắng tăng dần')
plt.tight_layout(); plt.show()
play(noisy_example, f'5 dB · reference: "{sub.iloc[0]["text"].lower()}"')
table(pd.DataFrame([('wav2vec2', noisy_pair[0].lower()), ('Whisper', noisy_pair[1])], columns=['model', 'transcript ở 5 dB']), 'Hai model nghe câu trên')
check('nhiễu 0 dB làm WER tăng với cả hai model', all(noise.iloc[-1, 1:] > noise.iloc[0, 1:]))
r = noise.set_index('SNR')
note(f'Từ sạch tới 0 dB, WER của wav2vec2 đi từ {r.at["sạch", "wav2vec2"]:.0%} lên {r.at["0 dB", "wav2vec2"]:.0%}, của Whisper từ {r.at["sạch", "Whisper"]:.0%} lên {r.at["0 dB", "Whisper"]:.0%}. '
     + ('Whisper chịu nhiễu tốt hơn: nó học trên hàng trăm nghìn giờ âm thanh đủ loại, còn wav2vec2-960h chỉ được fine-tune trên 960 giờ sách nói.' if r.at['0 dB', 'Whisper'] < r.at['0 dB', 'wav2vec2'] else
        'Ở mức nhiễu này wav2vec2 lại giữ được tốt hơn; muốn kết luận về độ bền thì phải đo trên đúng loại nhiễu của dữ liệu thi.'))

## 9. Thời gian chạy

Whisper luôn pad âm thanh lên đủ 30 giây trước khi vào encoder, rồi decoder sinh từng token. wav2vec2 chạy một lần qua đúng độ dài clip. Thời gian của Whisper đã đo ở mục 6; wav2vec2 đo ở đây.

In [ ]:
tw = []
for x in clean['wave']:
    if device == 'cuda': torch.cuda.synchronize()
    t = time.time(); w2v_logprobs(x)
    if device == 'cuda': torch.cuda.synchronize()
    tw.append((len(x) / SR, time.time() - t))
tw = np.array(tw); th = np.array([(s, t) for sp, s, t in times if sp == 'test.clean'])
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.scatter(tw[:, 0], tw[:, 1] * 1000, s=16, color=C['blue'], label='wav2vec2')
ax.scatter(th[:, 0], th[:, 1] * 1000, s=16, color=C['clay'], label='Whisper base.en')
ax.set_xlabel('độ dài clip (giây)'); ax.set_ylabel('thời gian (ms)'); ax.set_yscale('log'); ax.legend(frameon=False)
ax.set_title('Thời gian chép lời từng câu test-clean')
plt.tight_layout(); plt.show()
sh, lo = th[th[:, 0] < 5], th[th[:, 0] > 12]
table(pd.DataFrame([('wav2vec2', tw[:, 1].mean() * 1000, tw[:, 1].sum() / tw[:, 0].sum()),
                    ('Whisper base.en', th[:, 1].mean() * 1000, th[:, 1].sum() / th[:, 0].sum())],
                   columns=['model', 'ms mỗi câu', 'giây chạy / giây âm thanh']), 'Tốc độ trên ' + ('GPU' if device == 'cuda' else 'CPU'), fmt={'ms mỗi câu': '{:.0f}', 'giây chạy / giây âm thanh': '{:.3f}'})
check('mọi câu đều đo được thời gian', len(tw) == len(clean) and len(th) == len(clean))
note(f'Câu trên 12 giây dài gấp {lo[:, 0].mean() / sh[:, 0].mean():.1f} lần câu dưới 5 giây, nhưng Whisper chỉ tốn gấp {lo[:, 1].mean() / sh[:, 1].mean():.1f} lần '
     f'({sh[:, 1].mean() * 1000:.0f} và {lo[:, 1].mean() * 1000:.0f} ms): encoder luôn nhận đủ 30 giây, phần tăng thêm đến từ decoder sinh nhiều token hơn. '
     'Vì vậy không ước lượng thời gian chạy tập test bằng cách chia tỉ lệ theo độ dài âm thanh.')

## 10. Tóm tắt

In [ ]:
aw = agg.set_index(['tập', 'model'])['corpus WER']
summary = [('frame / ký tự trong câu ví dụ', f'{T} / {U}'),
           ('CTC loss: quy hoạch động / F.ctc_loss', f'{real_dp:.3f} / {real_ref:.3f}'),
           ('WER thô → chuẩn hóa (Whisper, test-clean)', f'{w_raw:.0%} → {w_norm:.1%}'),
           ('corpus WER test-clean: wav2vec2 / Whisper', f'{aw[("test.clean", "wav2vec2")]:.1%} / {aw[("test.clean", "Whisper")]:.1%}'),
           ('corpus WER test-other: wav2vec2 / Whisper', f'{aw[("test.other", "wav2vec2")]:.1%} / {aw[("test.other", "Whisper")]:.1%}'),
           ('WER ở nhiễu 0 dB: wav2vec2 / Whisper', f'{r.at["0 dB", "wav2vec2"]:.0%} / {r.at["0 dB", "Whisper"]:.0%}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 3 đổi từ khác trong transcript sai và xem loss tăng bao nhiêu; ở mục 8 thay nhiễu trắng bằng một đoạn tiếng ồn thật (ví dụ ESC-50 ở bài 7.2); thay Whisper base.en bằng `openai/whisper-small.en`.